# Friday — Deploy Your Model (Recognize · Decide · Act)
Today your Thursday model does real work. **Everyone** runs *vision-only recognize-&-label* first — no arm, no waiting. **Track A** then takes the *same model* to the robot arm.

You built the hard part already. This notebook wires it together and gives you the two functions the arm calls: **`classify`** (what is this?) and **`is_target`** (is this today's target, for real?).

> **How to run this.** The recognize/label cells use only `onnxruntime` + `numpy` + `Pillow`, so they run **here in the browser**, on a laptop, or on the **arm's Pi** — same code everywhere. The live‑webcam cell is optional (needs a local camera). The **On the arm** section near the end is **terminal commands you run over SSH**, not notebook cells.

## 1. Load your model
Point at your **Thursday** quantized model and paste your **class order** exactly as Thursday printed it. Order = meaning: the model returns one score per class *by position*.

In [ ]:
import os, numpy as np, onnxruntime as ort
from PIL import Image

USERNAME = "maya-p"                                   # <-- your JupyterHub username

WED_DIR = os.path.expanduser("~/wednesday")
THU_DIR = os.path.expanduser("~/thursday")
MODEL_PATH = os.path.join(THU_DIR, "recognizer_int8.onnx")   # int8 only on the arm

# Your class order comes straight from the model you trained -- no hand-typing.
# Wednesday saved it inside recognizer.pt; we read it back so labels.txt is exact.
# (This matters because everyone's object set can differ -- shared cube/cylinder/panda
#  plus any objects you added at home -- so the order is personal to you.)
try:
    import torch
    CLASS_NAMES = torch.load(os.path.join(WED_DIR, "recognizer.pt"),
                             map_location="cpu")["classes"]
except Exception as e:
    print("Could not auto-read classes (", e, ") -- type yours below, alphabetical:")
    CLASS_NAMES = ["cube", "cylinder", "none", "panda"]      # <-- fallback: edit to yours

TARGET    = "cube"        # <-- the object you'll pick today (one of CLASS_NAMES, not "none")
THRESHOLD = 0.60          # confidence bar  (matches the arm's --conf)
MARGIN    = 0.15          # top1-top2 gap   (matches the arm's --margin)

print("You     :", USERNAME)
print("Model   :", MODEL_PATH)
print("Classes :", CLASS_NAMES)
print("Target  :", TARGET)

In [ ]:
# Start an ONNX Runtime session (CPU — same as the Pi) and sanity-check the shapes.
sess = ort.InferenceSession(MODEL_PATH, providers=["CPUExecutionProvider"])

dummy = np.random.randn(1, 3, 224, 224).astype(np.float32)
out = sess.run(None, {"image": dummy})[0]
n_out = out.shape[1]
print("model outputs", n_out, "scores per image; you listed", len(CLASS_NAMES), "classes")
assert n_out == len(CLASS_NAMES), (
    "Mismatch! The model has a different number of classes than CLASS_NAMES.\n"
    "Fix CLASS_NAMES to match Thursday's export exactly, in the same order."
)
print("✅ model + class list agree")


### 🔧 Package your two files for the arm
Track A needs **two** files, both named after you so everyone's can share one folder: `<username>_int8.onnx` and `<username>_labels.txt`. The labels are written straight from the class order above (read from your trained model), so there's nothing to hand-type. Run this, then copy both to the arm in step 6.

In [ ]:
import shutil
# Name both files after you, so every student's files coexist in ~/models/ on the arm.
model_out  = f"{USERNAME}_int8.onnx"
labels_out = f"{USERNAME}_labels.txt"

shutil.copy(MODEL_PATH, model_out)                    # unique copy of your int8 model
with open(labels_out, "w") as f:                      # one class per line, YOUR order
    f.write("\n".join(CLASS_NAMES) + "\n")

print("ready to deploy:", model_out, "+", labels_out)
print("--- " + labels_out + " ---")
print(open(labels_out).read())

## 2. Preprocess + classify one crop
The arm's camera plus a localizer **crop** each object and hand you the crop (a regular camera — no depth). Your job is only to name it. This is the **exact call the arm makes** — same 224×224 resize and ImageNet normalization you trained with on Wednesday.

In [ ]:
IMG_SIZE = 224
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess(pil):
    img = pil.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    x = np.asarray(img, dtype=np.float32) / 255.0     # 0..1  (like ToTensor)
    x = (x - MEAN) / STD                              # ImageNet normalize
    x = x.transpose(2, 0, 1)[None, ...]               # HWC -> 1,C,H,W
    return x.astype(np.float32)

def classify(pil):
    """Return (label, confidence, all_probs) for one PIL image (a crop)."""
    scores = sess.run(None, {"image": preprocess(pil)})[0][0]   # raw logits
    e = np.exp(scores - scores.max()); probs = e / e.sum()      # softmax -> confidence
    idx = int(probs.argmax())
    return CLASS_NAMES[idx], float(probs[idx]), probs

print("classify() ready.")


### 🔧 Your turn — classify a few of your own photos
Drop a few test photos into a `friday_tests/` folder (target object, a different object, an empty/`none` shot), then run this. Do the labels and confidences look right?

In [ ]:
TEST_FOLDER = "friday_tests"   # <-- put a few .jpg/.png here (or point at ~/thursday/new_tests)

if os.path.isdir(TEST_FOLDER):
    for f in sorted(os.listdir(TEST_FOLDER)):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            label, conf, _ = classify(Image.open(os.path.join(TEST_FOLDER, f)))
            print(f"{f:24s} -> {label:12s} {conf:.0%}")
else:
    print(f"Make a '{TEST_FOLDER}' folder and add a few photos, then re-run.")


## 3. The question the arm actually asks: *is this the target?*
Softmax **always** returns something — point the camera at an empty table and it will still name your best guess. Three things stop the arm grabbing thin air (this is the concept from the slide):
1. a trained **`none`** class,
2. a **confidence threshold** — the top score must clear a bar, and
3. a **confidence margin** — the top score must *beat the runner-up* by a gap (sure, and not torn between two objects).

In [ ]:
def is_target(pil, target_name):
    label, conf, probs = classify(pil)
    top2 = np.sort(probs)[::-1][:2]        # two highest probabilities
    margin = float(top2[0] - top2[1])      # gap between best and runner-up

    # 🔧 TODO: the arm should act ONLY when all three are true:
    #   - it's the target:  label == target_name   (done for you below)
    #   - confident enough:  conf is at least THRESHOLD
    #   - a clear winner:    margin is at least MARGIN
    sure_enough  = ____      # compare conf   to THRESHOLD
    clear_winner = ____      # compare margin to MARGIN

    hit = (label == target_name) and sure_enough and clear_winner
    return hit, label, conf, margin

print("is_target() ready — fill in the two blanks above.")


### 🔧 Your turn — test the three cases + tune
Run `is_target` on **(a)** a target photo, **(b)** a *different* object, and **(c)** an empty/`none` photo. You want **True** only for (a). If the empty shot sneaks through as True, raise `THRESHOLD`/`MARGIN` or add more `none` photos. If your real target gets rejected, lower them a little.

In [ ]:
# Point these at three of your photos (any that exist on disk):
PHOTOS = {
    "target object ": "friday_tests/target.jpg",
    "other object  ": "friday_tests/other.jpg",
    "empty / none   ": "friday_tests/empty.jpg",
}

for name, path in PHOTOS.items():
    if os.path.exists(path):
        hit, label, conf, margin = is_target(Image.open(path), TARGET)
        verdict = "TARGET ✓" if hit else "skip"
        print(f"{name} -> {label:10s} {conf:4.0%}  margin {margin:.2f}   {verdict}")
    else:
        print(f"{name} -> (no file at {path})")


## 4. Vision-only: recognize & label (everyone runs this)
Draw a labeled box on a still image — **green** when it's the target, **gray** otherwise. This is your **Track B** deliverable and your **Track A backup**: once this works, you have a demo no matter what the hardware does.

In [ ]:
from PIL import ImageDraw

def label_image(path, save_as="labeled.jpg"):
    pil = Image.open(path).convert("RGB")
    hit, label, conf, margin = is_target(pil, TARGET)
    W, H = pil.size
    color = (34, 197, 94) if hit else (148, 163, 184)   # green if target, else gray
    box = [int(W*0.06), int(H*0.06), int(W*0.94), int(H*0.94)]
    draw = ImageDraw.Draw(pil)
    draw.rectangle(box, outline=color, width=max(3, W // 120))
    text = f"{label} {conf:.0%}" + ("   <- TARGET" if hit else "")
    draw.rectangle([box[0], box[1], box[0] + 9*len(text) + 12, box[1] + 22], fill=color)
    draw.text((box[0] + 6, box[1] + 5), text, fill=(15, 27, 45))
    pil.save(save_as)
    print(("TARGET " if hit else "       ") + text + f"   (margin {margin:.2f})  -> {save_as}")
    return pil

# 🔧 Your turn: run it on a few photos and open the saved images.
# label_image("friday_tests/target.jpg")


### (Optional, on your own laptop) Live webcam label
Same idea, live. This needs a **local camera**, so it won't open a window on the browser hub — run it on your laptop if you have Python + a webcam. Press **q** to quit.

In [ ]:
# Optional: live vision-only demo on a laptop with a webcam.
def webcam_label(threshold=None, camera_index=0):
    import cv2
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        print("No webcam here — use label_image() on saved photos instead.")
        return
    print("Press 'q' to quit.")
    while True:
        ok, frame = cap.read()
        if not ok: break
        pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        hit, label, conf, margin = is_target(pil, TARGET)
        color = (0, 200, 0) if hit else (0, 0, 255)
        cv2.putText(frame, f"{label} {conf:.0%}" + ("  TARGET" if hit else ""),
                    (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        cv2.imshow("recognize & label (q to quit)", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"): break
    cap.release(); cv2.destroyAllWindows()

# webcam_label()


## 5. What the arm does with your model
`vision_grasp.py` (the scaffold you tested Monday) runs this loop — the new part is *your* model:
1. A **localizer** (YOLO) answers **WHERE**: it finds an object and hands the arm a **crop**.
2. Your **classifier** answers **WHAT**: softmax → (label, confidence), plus the top1−top2 **margin** — the same numbers you computed above.
3. The **gate**: act only when the label is your **target**, confidence ≥ threshold, **and** margin ≥ the gap. That's exactly your `is_target`.
4. If it passes, the arm maps the object's **pixel** to a spot on the **table** using calibration — this works because the object is sitting on the table at a known height, so no depth camera is needed — and **picks** it. If nothing passes, it does nothing — which is correct.

## 6. On the arm (Track A) — run these in a **terminal**, not here
Copy your two files into the shared **`~/models/`** folder, then log in (address + password on the whiteboard). Because they're named after you, everyone's files live there at once — **no swapping between rotations**.

```bash
# from a JupyterHub Terminal (or the lab PC):
scp maya-p_int8.onnx maya-p_labels.txt pi@<arm>:~/models/

ssh pi@<arm>
cd ~                               # vision_grasp.py lives in the home directory
```

Let the menu walk you through it — **dry run first (no motion)**, then the live pick. When it asks for the model and labels, give **your** files under `models/`:

```bash
python3 vision_grasp.py --menu     # student · models/maya-p_int8.onnx · models/maya-p_labels.txt · dry-run = YES
# read the GATE line: does your target pass? (right label, conf over the bar, clear margin)

python3 vision_grasp.py --menu     # same choices, dry-run = NO   ->  hand near the stop
```

Prefer typing flags?

```bash
python3 vision_grasp.py --classifier student \
    --model models/maya-p_int8.onnx --labels models/maya-p_labels.txt \
    --target cube --preprocess imagenet --conf 0.60 --margin 0.15 --dry-run
```

Keep **`--preprocess imagenet`** — it matches how you trained. The arm runs the **int8** model only (the fp32-vs-int8 comparison was Thursday's job). Its classifier does the **same softmax + threshold + margin** you wrote above, so if the GATE passes here, the model half is done.

## 7. Tune & debug — symptom → which layer to fix
| Symptom | It's usually… | Fix |
|---|---|---|
| Confident **wrong** label | class order or lighting | check `labels.txt` order matches Thursday; match training-photo lighting |
| Grabs at an **empty** table | threshold/margin too low, thin `none` | raise `THRESHOLD`/`MARGIN`; add `none` photos |
| Names it right but **reaches the wrong spot** | calibration | re-check the camera→arm mapping (mentor) |
| Reaches right but **misses the grip** | pick height / object size | adjust pick height |


## 8. Tracks — where this leads
- **Track A (arm):** your deliverable is a working **pick** — the arm grabs the object only when it matches `TARGET`.
- **Track B (edge):** your deliverable is the **benchmark table** from Thursday (size / speed / accuracy) **plus** this recognize-&-label demo running on the Pi. Same model, different question: *how small and fast can it run?*
- **Track C (your idea):** reuse `classify()` for whatever you dreamed up (pet detector, card reader, …). The recognize-&-label cell is your starting template.

## Challenges (if you finish early)
1. **Sort, don't just pick.** Change `TARGET` between runs and send each object to a different bin.
2. **Two objects.** Place two well-separated objects and have the arm pick only the target.
3. **Show your reasoning.** In `label_image`, also draw the runner-up class and the margin — a great thing to explain at the showcase.
4. **Speed check.** Time `classify()` over 50 crops on the Pi. A few FPS is plenty for pick-and-place — why?